In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from collections import Counter


sns.set(rc={'figure.figsize': [25,9]}, font_scale=1.2)
sns.set_style('whitegrid') 


# Column Description After Merging (Working Dataset)

| Column    | Meaning                                 |
| --------- | --------------------------------------- |
| `userId`  | Unique identifier for a user            |
| `movieId` | Unique identifier for a movie           |
| `rating`  | Rating given by the user to the movie   |
| `title`   | Movie title                             |
| `genres`  | Genres associated with the movie        |
| `tag`     | User-generated tag describing the movie |


# Read Datasets and Optimize datatypes

### **Run ONCE ONLY we will keep using parquet files instead**

In [2]:
# change data types for optimization (takes less memory)
ratings = pd.read_csv("../data/raw/ratings.csv", dtype={"userId": "int32", "movieId": "int32", "rating": "float32"})
ratings = ratings.drop(columns=["timestamp"])    # we don't need timestamp for our analysis
ratings.to_parquet("../data/raw_parquet/ratings.parquet", index=False)


movies = pd.read_csv("../data/raw/movies.csv", dtype={"movieId": "int32", "title": "string", "genres": "category"})
movies.to_parquet("../data/raw_parquet/movies.parquet", index=False)

tags = pd.read_csv("../data/raw/tags.csv", dtype={"userId": "int32", "movieId": "int32", "tag": "string"})
tags = tags.drop(columns=["timestamp"])   # we don't need timestamp for our analysis


### Tags is very large and has a lot of text so we will take a look at it before saving it so it doesn't fail

In [3]:
tags.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000072 entries, 0 to 2000071
Data columns (total 3 columns):
 #   Column   Non-Null Count    Dtype 
---  ------   --------------    ----- 
 0   userId   2000072 non-null  int32 
 1   movieId  2000072 non-null  int32 
 2   tag      2000055 non-null  string
dtypes: int32(2), string(1)
memory usage: 30.5 MB


In [4]:
tags

,userId,movieId,tag
0,22,26479,Kevin Kline
1,22,79592,misogyny
2,22,247150,acrophobia
3,34,2174,music
4,34,2174,weird
...,...,...,...
2000067,162279,90645,Rafe Spall
2000068,162279,91079,Anton Yelchin
2000069,162279,91079,Felicity Jones
2000070,162279,91658,Rooney Mara


In [5]:
tags.isna().sum()

userId      0
movieId     0
tag        17
dtype: int64

In [6]:
# some movies have multiple tags, we need to combine them into a single row per movie
tags_combined = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.fillna(""))).reset_index()   # some movies have no tags
tags_combined

,movieId,tag
0,1,children Disney animation children Disney Disn...
1,2,Robin Williams fantasy Robin Williams time tra...
2,3,comedinha de velhinhos engraÃƒÂ§ada comedinha ...
3,4,characters slurs based on novel or book chick ...
4,5,Fantasy pregnancy remake family Steve Martin s...
...,...,...
51318,292143,Cadaqués catalonia China housing estate husban...
51319,292349,politically incorrect
51320,292371,Stephen King
51321,292597,artificial intelligence


In [7]:
tags_combined.info(show_counts=True)  #no nulls

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51323 entries, 0 to 51322
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  51323 non-null  int32 
 1   tag      51323 non-null  object
dtypes: int32(1), object(1)
memory usage: 601.6+ KB


In [8]:
tags_combined["tags"] = tags_combined["tag"].astype("string")     # Reduce memory usage
tags_combined.drop(columns=["tag"], inplace=True)      # tags is a better name

In [9]:
tags_combined.info(show_counts=True)  #no nulls and takes up less memory now

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51323 entries, 0 to 51322
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  51323 non-null  int32 
 1   tags     51323 non-null  string
dtypes: int32(1), string(1)
memory usage: 601.6 KB


In [10]:
tags_combined.to_parquet("../data/raw_parquet/tags.parquet", index=False)    # save the optimized tags data


### **End of Run ONCE ONLY section**

# Read Our optimized parquet files and start working 

In [4]:
ratings = pd.read_parquet("../data/raw_parquet/ratings.parquet")

movies = pd.read_parquet("../data/raw_parquet/movies.parquet")

tags = pd.read_parquet("../data/raw_parquet/tags.parquet")

In [5]:
ratings.info(show_counts= True)  # let's check each individual data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 3 columns):
 #   Column   Non-Null Count     Dtype  
---  ------   --------------     -----  
 0   userId   32000204 non-null  int32  
 1   movieId  32000204 non-null  int32  
 2   rating   32000204 non-null  float32
dtypes: float32(1), int32(2)
memory usage: 366.2 MB


In [6]:
ratings.sample(10)

,userId,movieId,rating
7605665,47592,318,5.0
28370488,177805,358,4.0
23703841,148498,5418,3.5
5462019,34075,27839,4.0
15735041,98636,6281,3.0
15487068,97024,6953,3.5
23643907,148102,187717,5.0
30912065,194070,3896,3.5
13938151,87106,2028,4.0
12677049,79374,2186,3.0


In [7]:
ratings.duplicated().sum()

np.int64(0)

In [8]:
movies.info(show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   movieId  87585 non-null  int32   
 1   title    87585 non-null  string  
 2   genres   87585 non-null  category
dtypes: category(1), int32(1), string(1)
memory usage: 1.2 MB


In [9]:
movies.sample(10)

,movieId,title,genres
71716,231411,Audrey Hepburn: Remembered (1993),Documentary
42326,162734,Sole Proprietor (2016),Action|Crime|Drama
60853,201909,The Night Visitor (2015),Horror
18781,98013,"Details, The (2011)",Comedy
19498,101339,Snowbeast (1977),Horror
63910,208877,The Captain from Köpenick (1931),Comedy
7325,7475,Raid (2003),Action|Crime|Drama|Thriller
63111,207037,Beast of the Water (2017),Adventure
71351,229947,The Division (2020),Action|Crime|Thriller
32715,141004,Victor Frankenstein (2015),Drama|Horror|Sci-Fi


In [10]:
movies.duplicated().sum()

np.int64(0)

In [11]:
tags.info(show_counts= True)  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51323 entries, 0 to 51322
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  51323 non-null  int32 
 1   tags     51323 non-null  string
dtypes: int32(1), string(1)
memory usage: 601.6 KB


In [12]:
tags.sample(10)

,movieId,tags
42047,184811,bikers dream lust madness murder priest rape s...
18522,104211,funny funny moments Jennifer Aniston funny mom...
11097,53769,great dialogue netflix atmospheric loneliness ...
27575,139064,La bande des quatre long
27881,140082,woman director
13340,72822,rwanda
27297,138006,1970s disco disco dancing montreal
27783,139747,one night romantic love manhattan new york cit...
3155,3448,Robin Williams Robin Williams Vietnam War Robi...
25684,132868,manslaughter press agent


In [13]:
tags.duplicated().sum()

np.int64(0)

### We need two files one for movies content and one for their ratings so will merge movie names with tags and save them 

In [14]:
movie_content = movies.merge(tags, on="movieId", how="left")   # we want all movie info including their tags


In [15]:
def fix_title(title):                     # fix titles like "Matrix, The" to "The Matrix" (with articles at the end)
    for article in ["The", "A", "An"]:     
        if f", {article}" in title:
            parts = title.split(f", {article}", 1)
            title = f"{article} {parts[0]}{parts[1]}"
            break
    return title


movie_content['title'] = movie_content['title'].apply(fix_title)


In [16]:
movie_content.sample(10)

,movieId,title,genres,tags
17915,93529,Submission (Underkastelsen) (2010),Documentary,<NA>
72282,233929,Don't Look Now: Looking Back (2002),Documentary,<NA>
51911,182853,"How Viktor ""The Garlic"" Took Alexey ""The Stud""...",Drama,foreign russian black humor road movie russian...
65435,212633,Snuff Bottle Connection (1977),Action,<NA>
22943,116549,Soldier of Fortune (1976),Adventure|Comedy|Drama,adventure knights very funny fistfight italy o...
73319,238164,In Broad Daylight (1991),Drama|Thriller,<NA>
82939,278264,Pathala Bhairavi (1951),Drama|Fantasy,<NA>
69995,225401,The Butterfly Effect (1995),Comedy|Romance,aunt incest older woman younger man relationship
56273,192153,Blue Spring Ride (2014),Drama|Romance,based on manga japanese
64842,211299,Bleed (2002),Horror,Full Moon Features


In [17]:
movie_content.info(show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   movieId  87585 non-null  int32   
 1   title    87585 non-null  object  
 2   genres   87585 non-null  category
 3   tags     51323 non-null  string  
dtypes: category(1), int32(1), object(1), string(1)
memory usage: 1.9+ MB


### Dropping rows with no tags will make us lose ratings which are actually more important, and it's possible there were no user given tags for movie since it's just extra info added by the community, so we'll just leave an empty string if it doesn't exist

In [18]:
movie_content["tags"] = movie_content["tags"].fillna("")  
movie_content.sample(20)

,movieId,title,genres,tags
34748,145506,24 Days (2014),Drama|Thriller,Race against time
29174,133077,A Romermed to the Teeth (1976),Action,
22789,116185,Thirty Day Princess (1934),Comedy|Romance,impersonation loan
25237,123213,Lease of Life (1954),(no genres listed),
24619,121847,Pickup Alley (1957),Action|Crime|Drama|Thriller,drug dealing BD-R
53922,187041,"Stockholm, My Love (2016)",Documentary|Drama,stockholm
54739,188753,Unfriended: Dark Web (2018),Horror,facebook found footage social media boring dar...
44064,166383,Sofía de Niño Rivera: Exposed (2016),Comedy,stand-up comedy
61829,204052,The Informer (1929),Drama,
37489,152037,Grease Live (2016),(no genres listed),live performance live television musical based...


In [19]:
movie_content.info(show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   movieId  87585 non-null  int32   
 1   title    87585 non-null  object  
 2   genres   87585 non-null  category
 3   tags     87585 non-null  string  
dtypes: category(1), int32(1), object(1), string(1)
memory usage: 1.9+ MB


In [20]:
movie_content.isna().sum()

movieId    0
title      0
genres     0
tags       0
dtype: int64

In [21]:
movie_content.duplicated().sum()

np.int64(0)

# Save our cleaned data as two parquet files for faster reading and smaller size (Easier and Faster for both Analysis and ML)

### We will save two separate files, one for movie content and one for ratings. This ensures files take less memory and we can combine/ merge them when needed after loading them separately. 

In [22]:
movie_content.to_parquet("../data/clean/movie_content_clean.parquet", index= False)   # saving them separately for memory optimization
ratings.to_parquet("../data/clean/ratings_clean.parquet", index= False)

# Merge the two dfs to get the whole picture and start our analysis

In [23]:
df = ratings.merge( movie_content, on="movieId", how="left")
df.info(show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 6 columns):
 #   Column   Non-Null Count     Dtype   
---  ------   --------------     -----   
 0   userId   32000204 non-null  int32   
 1   movieId  32000204 non-null  int32   
 2   rating   32000204 non-null  float32 
 3   title    32000204 non-null  object  
 4   genres   32000204 non-null  category
 5   tags     32000204 non-null  string  
dtypes: category(1), float32(1), int32(2), object(1), string(1)
memory usage: 915.6+ MB


In [24]:
df.sample(10)

,userId,movieId,rating,title,genres,tags
1764660,11191,70728,3.5,Bronson (2009),Action|Comedy|Drama|Thriller,mental illness psychological prison psychologi...
21311272,133346,2717,4.5,Ghostbusters II (1989),Comedy|Fantasy|Sci-Fi,comedy Bill Murray Dan Aykroyd ghosts comedy g...
4497832,28143,194016,4.5,Ralph Breaks the Internet (2018),Animation|Children,Disney friendship Pixar internet wifi funny po...
1399667,8905,77709,4.0,The Sky Crawlers (Sukai kurora) (2008),Adventure|Animation|Drama,anime dogfights drama pilots war anime forgett...
13552656,84709,1094,5.0,The Crying Game (1992),Drama|Romance|Thriller,Forest Whitaker male nudity Miranda Richardson...
2795008,17636,356,4.0,Forrest Gump (1994),Comedy|Drama|Romance|War,bittersweet comedy drama emotional great actin...
27450777,172199,1653,4.0,Gattaca (1997),Drama|Sci-Fi|Thriller,bioethics dystopia sci-fi beautiful drama euge...
16482267,103252,4597,1.5,Gleaming the Cube (1989),Action|Drama|Mystery,Action Eighties Skateboarding action car chase...
24044991,150720,3255,3.0,A League of Their Own (1992),Comedy|Drama,Geena Davis feminism Rosie O'Donnell seen more...
2109570,13443,5903,4.0,Equilibrium (2002),Action|Sci-Fi|Thriller,Christian Bale post-apocalyptic revolution sty...


In [25]:
df.isna().sum()

userId     0
movieId    0
rating     0
title      0
genres     0
tags       0
dtype: int64

In [26]:
df.duplicated().sum()

np.int64(0)

# Analysis

In [27]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
userId,32000204.0,100278.506411,57949.046233,1.0,50053.0,100297.0,150451.0,200948.0
movieId,32000204.0,29318.610122,50958.160880,1.0,1233.0,3452.0,44199.0,292757.0
rating,32000204.0,3.540395,1.058986,0.5,3.0,3.5,4.0,5.0


In [28]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
userId,32000204.0,NaN,NaN,NaN,100278.506411,57949.046233,1.0,50053.0,100297.0,150451.0,200948.0
movieId,32000204.0,NaN,NaN,NaN,29318.610122,50958.16088,1.0,1233.0,3452.0,44199.0,292757.0
rating,32000204.0,NaN,NaN,NaN,3.540395,1.058986,0.5,3.0,3.5,4.0,5.0
title,32000204,84236,The Shawshank Redemption (1994),102929,NaN,NaN,NaN,NaN,NaN,NaN,NaN
genres,32000204,1783,Drama,2256325,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tags,32000204,43230,,175367,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. What are the most common movie genres rated by users?

In [29]:
top_genres = ratings.merge( movie_content[['movieId', 'genres']], on='movieId', how='left')   # get genres df we'll use for genre related analysis (all of them)

top_genres = top_genres.dropna()

top_genres = top_genres.assign(genre = top_genres['genres'].str.split('|')).explode('genre')

top_genres.sample(15)

,userId,movieId,rating,genres,genre
18970529,118736,6333,3.0,Action|Adventure|Sci-Fi|Thriller,Adventure
28108051,176158,5479,3.0,Action|Adventure|Drama|Thriller,Adventure
7591248,47549,293,4.5,Action|Crime|Drama|Thriller,Action
3222438,20284,780,4.0,Action|Adventure|Sci-Fi|Thriller,Thriller
21879311,136903,103141,2.5,Adventure|Animation|Comedy,Adventure
4583840,28683,914,2.5,Comedy|Drama|Musical|Romance,Romance
13878429,86768,27706,5.0,Adventure|Children|Comedy|Fantasy,Children
30453691,190989,3516,4.0,Comedy|Fantasy|Romance,Romance
26643744,167466,587,4.0,Comedy|Drama|Fantasy|Romance|Thriller,Comedy
22498087,140922,2797,4.5,Comedy|Drama|Fantasy|Romance,Romance


In [30]:
top_genres_count = (top_genres['genre'].value_counts().reset_index(name='num_ratings_genre').head(10))
top_genres_count

,genre,num_ratings_genre
0,Drama,13973271
1,Comedy,11206926
2,Action,9665213
3,Thriller,8679464
4,Adventure,7590522
5,Sci-Fi,5717337
6,Romance,5524615
7,Crime,5373051
8,Fantasy,3702759
9,Children,2731841


In [31]:
px.bar(top_genres_count, x='genre', y='num_ratings_genre', title='Top 10 Movie Genres by Number of Ratings',
       labels={'genre': 'Genre', 'num_ratings_genre': 'Number of Ratings per Genre'}, height=650, width=1500, color_discrete_sequence=['royalblue'])

## 2. What are the genres most liked by users? (Which genres have the highest average ratings?)


In [32]:
top_genres_avg = top_genres.groupby('genre')['rating'].mean().reset_index(name='avg_rating_genre').sort_values(by='avg_rating_genre', ascending=False).head(10)
top_genres_avg

,genre,avg_rating_genre
10,Film-Noir,3.915774
18,War,3.791699
6,Crime,3.691771
7,Documentary,3.691181
8,Drama,3.682454
14,Mystery,3.673103
3,Animation,3.615332
19,Western,3.600175
12,IMAX,3.593313
13,Musical,3.554277


In [33]:
px.bar(top_genres_avg, x='genre', y='avg_rating_genre', title='Top 10 Movie Genres by Average User Rating', labels={'genre': 'Genre', 'avg_rating_genre': 'Average Rating per Genre'}, height=650, width=1500)

## 3. What are the top 10 most liked movies? (Which movies have the highest average ratings? (min 1000 ratings per movie))

In [34]:
top_movies = ratings.merge(movie_content[['movieId', 'title']],on='movieId',how='left')  
top_movies.dropna(inplace=True)
top_movies.sample(15)

,userId,movieId,rating,title
20266224,126921,8754,3.5,The Prime of Miss Jean Brodie (1969)
28039501,175638,90439,4.0,Margin Call (2011)
27084643,170164,30707,3.5,Million Dollar Baby (2004)
29672699,186085,1254,3.5,The Treasure of the Sierra Madre (1948)
22720830,142331,244756,0.5,The Girl Who Believes in Miracles (2021)
20797925,130208,8376,3.5,Napoleon Dynamite (2004)
10110131,63292,899,3.0,Singin' in the Rain (1952)
8355955,52352,2,2.0,Jumanji (1995)
28568255,179039,77307,5.0,Dogtooth (Kynodontas) (2009)
15964673,100044,22,4.0,Copycat (1995)


In [35]:
min_num= 1000   # at least 1000 ratings to be considered liked 

top_movies_avg = top_movies.groupby('title').agg(avg_rating_movie=('rating', 'mean'), num_ratings_movie=('rating', 'count')).reset_index().query('num_ratings_movie >= @min_num').sort_values(by='avg_rating_movie', ascending=False).head(10)
top_movies_avg


,title,avg_rating_movie,num_ratings_movie
48929,Planet Earth II (2016),4.446830,1956
48928,Planet Earth (2006),4.444369,2948
7714,Band of Brothers (2001),4.426538,2811
72900,The Shawshank Redemption (1994),4.404614,102929
66166,The Godfather (1972),4.317030,66440
47696,Parasite (2019),4.312253,11670
10359,Blue Planet II (2017),4.300086,1163
78476,Twin Peaks (1989),4.298684,1140
250,12 Angry Men (1957),4.265311,21863
74818,The Usual Suspects (1995),4.265070,67750


In [36]:
px.bar(top_movies_avg, x='title', y='avg_rating_movie', title='Top 10 Movies by Average User Rating (Min 1000 Ratings)', labels={'title': 'Movie Title', 'avg_rating_movie': 'Average Rating per Movie'}, height=650, width=1500) 

## 4. What are the top 10 most popular movies? (Which movies have the highest number of ratings?)

In [37]:
top_movies_count = top_movies['title'].value_counts().reset_index(name='num_ratings_movie').head(10)
top_movies_count

,title,num_ratings_movie
0,The Shawshank Redemption (1994),102929
1,Forrest Gump (1994),100296
2,Pulp Fiction (1994),98409
3,The Matrix (1999),93808
4,The Silence of the Lambs (1991),90330
5,Star Wars: Episode IV - A New Hope (1977),85010
6,Fight Club (1999),77332
7,Jurassic Park (1993),75233
8,Schindler's List (1993),73849
9,The Lord of the Rings: The Fellowship of the R...,73122


In [38]:
px.bar(top_movies_count, x='title', y='num_ratings_movie', title='Top 10 Movies by Number of Ratings',
       labels={'title': 'Movie Title', 'num_ratings_movie': 'Number of Ratings per Movie'}, height=650, width=1500, color_discrete_sequence=['royalblue'])

## 5. How many ratings per user for top 100,000 users? (User Activity Distribution for top 100,000 users)

In [39]:
ratings_user_count = ratings.groupby('userId')['rating'].count().reset_index(name='num_ratings_user').sort_values(by='num_ratings_user', ascending=False).head(100_000)
ratings_user_count

,userId,num_ratings_user
175324,175325,33332
17034,17035,9577
55652,55653,9178
123464,123465,9044
171794,171795,9016
...,...,...
51268,51269,73
106589,106590,73
92513,92514,73
129414,129415,73


In [40]:
fig= px.histogram(ratings_user_count, x='num_ratings_user', nbins=30,
                  title='User Activity Distribution for Top 100,000 Users (Number of Ratings per User)',labels={'num_ratings_user':'Number of Ratings'}, color_discrete_sequence=['royalblue'])
fig.update_yaxes(title_text='Number of Users')

fig.show()

## 6. Do more active users give higher or lower ratings on average? (Average Rating vs Number of Ratings per User for Top 100,00 Users)

In [41]:
ratings_user = ratings.groupby('userId')['rating'].agg(num_ratings_user='count', avg_rating_user='mean').reset_index().sort_values(by='num_ratings_user', ascending=False).head(100_000)
ratings_user

,userId,num_ratings_user,avg_rating_user
175324,175325,33332,3.077808
17034,17035,9577,2.567819
55652,55653,9178,3.280290
123464,123465,9044,2.528859
171794,171795,9016,3.181954
...,...,...,...
51268,51269,73,3.794521
106589,106590,73,3.390411
92513,92514,73,3.773973
129414,129415,73,3.883562


In [42]:
px.scatter(ratings_user,x='num_ratings_user',y='avg_rating_user',title='Do More Active Users Give Higher or Lower Ratings on Average? (Average Rating vs Number of Ratings per User for Top 100,00 Users)',
           labels={'num_ratings_user':'Number of Ratings per User', 
                'avg_rating_user':'Average Rating per User'}, trendline='ols', color_discrete_sequence=['royalblue'], opacity=0.3)


## 7. Which tags are most frequently associated with highly rated movies? (Top 40 Tags for Movies Rated 4.0 and Above)


In [43]:
movies_avg = ratings.groupby('movieId')['rating'].mean().reset_index(name='avg_rating_movie')
movies_high_rated = movies_avg[movies_avg['avg_rating_movie'] >= 4.0]
movies_high_rated.sample(15)

,movieId,avg_rating_movie
35631,150696,4.189655
84303,291831,5.000000
49749,183023,5.000000
18403,96516,4.000000
30187,137178,5.000000
74010,255493,4.500000
34087,146650,4.333333
21297,110163,4.000000
64086,215385,4.000000
1860,1949,4.099410


In [44]:
tags_high_rated = movies_high_rated.merge(movie_content[['movieId','tags']], on='movieId', how='left')
tags_high_rated.sample(15)       # show some highly rated movies with their tags

,movieId,avg_rating_movie,tags
1161,141958,4.05,
2883,190345,4.00,doomed love
3974,216241,4.00,
7132,290854,5.00,
4870,238794,5.00,
5039,243918,4.00,
3606,207802,4.00,
1877,165649,4.00,college sorority
3243,198449,4.00,
4960,241452,4.00,


In [45]:
tags_high_rated_top = tags_high_rated['tags'].value_counts().head(40).reset_index()
tags_high_rated_top     # show top 40 tags for highly rated movies

,tags,count
0,,4960
1,woman director,91
2,stand-up comedy,12
3,independent film,10
4,time travel,9
5,found footage haunting,8
6,cimrman theatre play,7
7,tivo21,6
8,musical,6
9,documentary,6


In [46]:
fig= px.bar(tags_high_rated_top, x='tags', y='count',
            title='Top 40 Tag Strings Associated with Highly Rated Movies (Rating >= 4.0)',labels={'tags':'Tag Text','count':'Frequency'}, width=1550, height=850, color_discrete_sequence=['darkorange'])
fig.update_xaxes(tickangle=45, automargin=True)
fig.update_yaxes(type='log')
fig.show()

## 8. How many ratings do movies typically receive? (How many movies receive a given number of ratings)

In [47]:
ratings_movie_num = (ratings.groupby('movieId')['rating'].count().reset_index(name='num_ratings_movie'))
ratings_movie_num.sample(15)

,movieId,num_ratings_movie
38381,157599,3
38521,157957,6
23703,119814,1
21793,112650,13
47748,178649,20
37635,155559,2
7793,8450,22
28720,133771,5433
30526,138028,2
33409,144920,8


In [48]:
fig = px.histogram(ratings_movie_num, x='num_ratings_movie', nbins=30, title='Distribution of Number of Ratings per Movie',
                   labels={'num_ratings_movie': 'Number of Ratings per Movie'},color_discrete_sequence=['royalblue'], width=1500, height=600)

fig.update_yaxes(title_text='Number of Movies with Given Number of Ratings ')
fig.update_yaxes(type='log')
fig.show()

## 9. Do some genres receive higher or more consistent ratings? (Distribution of ratings per top 10 genres)

In [49]:
# use previously created top_genres in our first question in our analysis and create a new series for top 10 genres
top_genres 


,userId,movieId,rating,genres,genre
0,1,17,4.0,Drama|Romance,Drama
0,1,17,4.0,Drama|Romance,Romance
1,1,25,1.0,Drama|Romance,Drama
1,1,25,1.0,Drama|Romance,Romance
2,1,29,2.0,Adventure|Drama|Fantasy|Mystery|Sci-Fi,Adventure
...,...,...,...,...,...
32000200,200948,79796,1.0,Action|Adventure|Drama|Thriller|War,Thriller
32000200,200948,79796,1.0,Action|Adventure|Drama|Thriller|War,War
32000201,200948,80350,0.5,Comedy,Comedy
32000202,200948,80463,3.5,Drama,Drama


In [50]:
top_10_genres = top_genres['genre'].value_counts().head(10).index
top_10_genres

Index(['Drama', 'Comedy', 'Action', 'Thriller', 'Adventure', 'Sci-Fi',
       'Romance', 'Crime', 'Fantasy', 'Children'],
      dtype='object', name='genre')

In [51]:
top_genres_distribution = top_genres[top_genres['genre'].isin(top_10_genres)]
top_genres_distribution

,userId,movieId,rating,genres,genre
0,1,17,4.0,Drama|Romance,Drama
0,1,17,4.0,Drama|Romance,Romance
1,1,25,1.0,Drama|Romance,Drama
1,1,25,1.0,Drama|Romance,Romance
2,1,29,2.0,Adventure|Drama|Fantasy|Mystery|Sci-Fi,Adventure
...,...,...,...,...,...
32000200,200948,79796,1.0,Action|Adventure|Drama|Thriller|War,Drama
32000200,200948,79796,1.0,Action|Adventure|Drama|Thriller|War,Thriller
32000201,200948,80350,0.5,Comedy,Comedy
32000202,200948,80463,3.5,Drama,Drama


In [52]:
# take a sample for better visualization and performance
distribution_sample = top_genres_distribution.sample(n=100000, random_state=42)
distribution_sample

,userId,movieId,rating,genres,genre
4169574,26174,4149,2.0,Comedy|Romance,Comedy
10226330,64056,4226,3.5,Mystery|Thriller,Thriller
30736476,192899,5782,5.0,Action|Drama|Thriller,Action
12084087,75593,1921,3.0,Drama|Sci-Fi|Thriller,Thriller
18136486,113406,2496,5.0,Comedy|Romance,Romance
...,...,...,...,...,...
11426037,71417,260,4.0,Action|Adventure|Sci-Fi,Sci-Fi
6009646,37562,143385,4.0,Drama|Thriller,Drama
22555971,141289,2294,3.5,Adventure|Animation|Children|Comedy|Fantasy,Comedy
7679894,48074,1291,4.0,Action|Adventure,Action


In [ ]:
px.box(distribution_sample,x='genre',y='rating',title='Rating Distribution by Genre (Top 10 Genres)',
       labels={'genre':'Genre', 'rating':'Rating'}, color_discrete_sequence=['royalblue'], width=1500, height=600)

# End of Notebook